In [257]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
from datetime import date, datetime


In [258]:
def make_soup(url: str) -> BeautifulSoup:
    source_code = requests.get(url, allow_redirects=False)
    plain_text = source_code.text.encode("ascii", "replace")
    return BeautifulSoup(plain_text, "html.parser")

def get_all_event_details(soup_obj):

    all_event_links = []

    for link in soup_obj.find_all("td", {"class": "b-statistics__table-col"}):
                for href in link.find_all("a"):
                    event_link = href.get("href")
                    all_event_links.append(event_link)

    return all_event_links

def get_all_fight_details(event_soup_obj):
      
    fight_details_links = []

    for fight_detail in event_soup_obj.find_all("tr", {"class": "b-fight-details__table-row b-fight-details__table-row__hover js-fight-details-click"}):
        href = fight_detail.get("data-link")
        fight_details_links.append(href)

    return fight_details_links


In [259]:
all_events_url="http://ufcstats.com/statistics/events/completed?page=all"

test_soup = make_soup(all_events_url)

all_event_links = get_all_event_details(test_soup)

test_event = all_event_links[1]

test_event_soup = make_soup(test_event)

fight_links = get_all_fight_details(test_event_soup)
fight_links[:5]

['http://ufcstats.com/fight-details/d69bd8e53ec858dc',
 'http://ufcstats.com/fight-details/721642e7de112e9e',
 'http://ufcstats.com/fight-details/ad487d1c6141d90a',
 'http://ufcstats.com/fight-details/fef80c98b2fba70c',
 'http://ufcstats.com/fight-details/c27bec45d5a29a2c']

In [260]:
fight_soup = make_soup(fight_links[0])


In [261]:
tables = fight_soup.findAll("tbody")
test_table = tables[0]


/var/folders/ft/jgrps1md2_30cc3nx_zr573r0000gn/T/ipykernel_45786/459022420.py:1: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  tables = fight_soup.findAll("tbody")


In [262]:
tables = fight_soup.findAll("tbody")
totals_table = tables[0]
totals_per_round_table = tables[1]
significant_strikes_table = tables[2]
significant_strikes_per_round_table = tables[3]


/var/folders/ft/jgrps1md2_30cc3nx_zr573r0000gn/T/ipykernel_45786/4181895494.py:1: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  tables = fight_soup.findAll("tbody")


In [263]:
type(totals_per_round_table.find('tr'))
# type(totals_per_round_table.find_all('tr'))

bs4.element.Tag

In [264]:
len(totals_per_round_table.find_all('tr'))

2

In [265]:
test_sig_str_data = significant_strikes_table.find('tr').find_all('td')
for i in test_sig_str_data:
    print(i.text.replace("\n    ",""))



Renato Moicano 


Chris Duncan 



  21 of 36
  27 of 62


  58%
  43%


  16 of 31
  8 of 39


  4 of 4
  5 of 7


  1 of 1
  14 of 16


  20 of 34
  25 of 60


  1 of 1
  2 of 2


  0 of 1
  0 of 0



In [266]:
totals_table_structure = {0: ['fighters', '\n'],
                          1: ['knockdowns', '\n    '],
                          2: ['significant strikes', '\n    '],
                          3: ['significant strike percentage', '\n    '],
                          4: ['total strike', '\n    '],
                          5: ['takedowns', '\n    '],
                          6: ['takedown percentage', '\n    '],
                          7: ['submission attempts', '\n    '],
                          8: ['reversals', '\n    '],
                          9: ['control', '\n    ']}

sig_str_table_structure = {0: ['fighters', '\n'],
                          1: ['significant strikes', '\n    '],
                          2: ['significant strike percentage', '\n    '],
                          3: ['head', '\n    '],
                          4: ['body', '\n    '],
                          5: ['leg', '\n    '],
                          6: ['distance', '\n    '],
                          7: ['clinch', '\n    '],
                          8: ['ground', '\n    ']}


In [267]:
## get all of the table data and initialize the data dictionary
totals_dat = totals_table.find_all('td')
test_total_dict = {}

for i in range(len(totals_dat)):
    ## remove new lines and tabs for easier formatting
    table_row_data_clean = totals_dat[i].text.replace("\n\n", "").replace("      ","").replace("\n    \n","")

    ## fill in the dictioanry for the corresponding structure
    test_total_dict[totals_table_structure[i][0]] = table_row_data_clean.split(totals_table_structure[i][1])

print(test_total_dict)

{'fighters': ['Renato Moicano ', 'Chris Duncan '], 'knockdowns': ['0', '0'], 'significant strikes': ['21 of 36', '27 of 62'], 'significant strike percentage': ['58%', '43%'], 'total strike': ['62 of 84', '30 of 66'], 'takedowns': ['1 of 4', '0 of 2'], 'takedown percentage': ['25%', '0%'], 'submission attempts': ['1', '0'], 'reversals': ['0', '0'], 'control': ['3:16', '0:01']}


In [268]:
# totals_table
def parse_total_table(table, table_structure):

    ## get all of the table data and initialize the data dictionary
    table_data = table.find_all('td')
    table_dict = {}

    for i in range(len(table_data)):
        ## remove new lines and tabs for easier formatting
        table_data_clean = table_data[i].text.replace("\n\n", "").replace("      ","").replace("\n    \n","")

        ## fill in the dictioanry for the corresponding structure
        table_dict[table_structure[i][0]] = table_data_clean.split(table_structure[i][1])

    return table_dict

# parse_total_table(totals_table,totals_table_structure)
parse_total_table(significant_strikes_table,sig_str_table_structure)



{'fighters': ['Renato Moicano ', 'Chris Duncan '],
 'significant strikes': ['21 of 36', '27 of 62'],
 'significant strike percentage': ['58%', '43%'],
 'head': ['16 of 31', '8 of 39'],
 'body': ['4 of 4', '5 of 7'],
 'leg': ['1 of 1', '14 of 16'],
 'distance': ['20 of 34', '25 of 60'],
 'clinch': ['1 of 1', '2 of 2'],
 'ground': ['0 of 1', '0 of 0']}

In [269]:
# def parse_per_round_table(per_round_table, per_round_table_structure):

#     ## Save all table headers and table rows:
#     headers_and_rows = per_round_table.find_all(['th','tr'])

#     ## Initialize the stats_per_round_dict and round unumber object
#     stats_per_round_dict = {}
#     round = None

#     for i in headers_and_rows:

#         ## if the soup_item is a table header, set our round variable to it
#         if i.name == 'th':
#             round_num = i.text.replace("\n              ", "").replace("\n            ","")
#             # print(round_num)

#         ## if the soup_item is a table row, parse the table row data and assign it to the current round
#         if i.name == 'tr':

#              stats_per_round_dict[round_num] = extract_data_from_table(i, per_round_table_structure)

#     return stats_per_round_dict

# # parse_per_round_table(totals_per_round_table,totals_table_structure)
# parse_per_round_table(significant_strikes_per_round_table,sig_str_table_structure)


In [270]:

## Save all table headers and table rows:
headers_and_rows = totals_per_round_table.find_all(['th','tr'])

## Initialize the stats_per_round_dict and round unumber object
stats_per_round_dict = {}
round = None

for i in headers_and_rows:

    ## if the soup_item is a table header, set our round variable to it
    if i.name == 'th':
        round_num = i.text.replace("\n              ", "").replace("\n            ","")
        # print(round_num)

    ## if the soup_item is a table row, parse the table row data and assign it to the current round
    if i.name == 'tr':
        round_row_data = i.find_all('td')

        ind_round_dict = {}

        for i in range(len(round_row_data)):

            ## remove new lines and tabs for easier formatting
            round_row_data_clean = round_row_data[i].text.replace("\n\n", "").replace("      ","").replace("\n    \n","")

            ## fill in the individual_round_dictioanry for the corresponding structure
            ind_round_dict[totals_table_structure[i][0]] = round_row_data_clean.split(totals_table_structure[i][1])

        stats_per_round_dict[round_num] = ind_round_dict
print(stats_per_round_dict.keys())
print(stats_per_round_dict)




dict_keys(['Round 1', 'Round 2'])
{'Round 1': {'fighters': ['Renato Moicano ', 'Chris Duncan '], 'knockdowns': ['0', '0'], 'significant strikes': ['14 of 20', '17 of 38'], 'significant strike percentage': ['70%', '44%'], 'total strike': ['16 of 22', '19 of 41'], 'takedowns': ['0 of 2', '0 of 1'], 'takedown percentage': ['0%', '0%'], 'submission attempts': ['0', '0'], 'reversals': ['0', '0'], 'control': ['1:13', '0:01']}, 'Round 2': {'fighters': ['Renato Moicano ', 'Chris Duncan '], 'knockdowns': ['0', '0'], 'significant strikes': ['7 of 16', '10 of 24'], 'significant strike percentage': ['43%', '41%'], 'total strike': ['46 of 62', '11 of 25'], 'takedowns': ['1 of 2', '0 of 1'], 'takedown percentage': ['50%', '0%'], 'submission attempts': ['1', '0'], 'reversals': ['0', '0'], 'control': ['2:03', '0:00']}}


In [ ]:
totals_table_structure = {0: ['fighters', '\n'],
                          1: ['knockdowns', '\n    '],
                          2: ['significant strikes', '\n    '],
                          3: ['significant strike percentage', '\n    '],
                          4: ['total strike', '\n    '],
                          5: ['takedowns', '\n    '],
                          6: ['takedown percentage', '\n    '],
                          7: ['submission attempts', '\n    '],
                          8: ['reversals', '\n    '],
                          9: ['control', '\n    ']}

sig_str_table_structure = {0: ['fighters', '\n'],
                          1: ['significant strikes', '\n    '],
                          2: ['significant strike percentage', '\n    '],
                          3: ['head', '\n    '],
                          4: ['body', '\n    '],
                          5: ['leg', '\n    '],
                          6: ['distance', '\n    '],
                          7: ['clinch', '\n    '],
                          8: ['ground', '\n    ']}

def parse_agg_table(table, table_structure):

    ## get all of the table data and initialize the data dictionary
    table_data = table.find_all('td')
    table_dict = {}

    for i in range(len(table_data)):
        ## remove new lines and tabs for easier formatting
        table_data_clean = table_data[i].text.replace("\n\n", "").replace("      ","").replace("\n    \n","").rstrip()

        ## fill in the dictioanry for the corresponding structure
        table_dict[table_structure[i][0]] = table_data_clean.split(table_structure[i][1])

    return table_dict

def parse_per_round_table(per_round_table, per_round_table_structure):

    ## Save all table headers and table rows:
    headers_and_rows = per_round_table.find_all(['th','tr'])

    ## Initialize the stats_per_round_dict and round unumber object
    stats_per_round_dict = {}
    round = None

    for i in headers_and_rows:

        ## if the soup_item is a table header, set our round variable to it
        if i.name == 'th':
            round_num = i.text.replace("\n              ", "").replace("\n            ","")
            # print(round_num)

        ## if the soup_item is a table row, parse the table row data and assign it to the current round
        if i.name == 'tr':

             stats_per_round_dict[round_num] = parse_agg_table(i, per_round_table_structure)

    return stats_per_round_dict

In [272]:
#function: 
#   seperate fight data into different sections  -- DONE
#       for each fight section:
#           extract the correct amount of tables (1 or 2) -- DONE
#   for each extracted table:
#       get the information using a pre-made dictioanry.  -- DONE
#       Append the resulting row to a csv

def extract_fight_data(fight_soup, totals_table_structure, sig_str_table_structure):

    fight_tables = fight_soup.find_all("tbody")
    fight_data = {}

    ## seperate fight data into different sections 
    totals_table = fight_tables[0]
    totals_per_round_tables = fight_tables[1]
    sig_str_table = fight_tables[2]
    sig_str_per_round_tables = fight_tables[3]

    ## combine per_round tables into a single table
    fight_data['fight_totals'] = parse_agg_table(totals_table, totals_table_structure)
    fight_data['totals_per_round'] = parse_per_round_table(totals_per_round_tables, totals_table_structure)

    ## sig_str table
    fight_data['sig_str'] = parse_agg_table(sig_str_table, sig_str_table_structure)
    fight_data['sig_str_per_round'] = parse_per_round_table(sig_str_per_round_tables, sig_str_table_structure)

    return fight_data

test_fight_stats = extract_fight_data(fight_soup, totals_table_structure, sig_str_table_structure)


In [273]:
test_fight_stats

{'fight_totals': {'fighters': ['Renato Moicano ', 'Chris Duncan '],
  'knockdowns': ['0', '0'],
  'significant strikes': ['21 of 36', '27 of 62'],
  'significant strike percentage': ['58%', '43%'],
  'total strike': ['62 of 84', '30 of 66'],
  'takedowns': ['1 of 4', '0 of 2'],
  'takedown percentage': ['25%', '0%'],
  'submission attempts': ['1', '0'],
  'reversals': ['0', '0'],
  'control': ['3:16', '0:01']},
 'totals_per_round': {'Round 1': {'fighters': ['Renato Moicano ',
    'Chris Duncan '],
   'knockdowns': ['0', '0'],
   'significant strikes': ['14 of 20', '17 of 38'],
   'significant strike percentage': ['70%', '44%'],
   'total strike': ['16 of 22', '19 of 41'],
   'takedowns': ['0 of 2', '0 of 1'],
   'takedown percentage': ['0%', '0%'],
   'submission attempts': ['0', '0'],
   'reversals': ['0', '0'],
   'control': ['1:13', '0:01']},
  'Round 2': {'fighters': ['Renato Moicano ', 'Chris Duncan '],
   'knockdowns': ['0', '0'],
   'significant strikes': ['7 of 16', '10 of 24']

In [274]:

chr(97)

'a'

In [275]:
def split_combined_data_rows(fight_table_dict):

    split_data_dict = {}

    test_fight_stats['sig_str_per_round']
    for key, value_list in test_fight_stats['fight_totals'].items():
        for i, val in enumerate(value_list):

            ## create new key based on inext
            new_key = f"{key}_{chr(97 + i)}"
            split_data_dict[new_key] = val

    return split_data_dict


In [276]:
print(test_fight_stats['fight_totals'])

{'fighters': ['Renato Moicano ', 'Chris Duncan '], 'knockdowns': ['0', '0'], 'significant strikes': ['21 of 36', '27 of 62'], 'significant strike percentage': ['58%', '43%'], 'total strike': ['62 of 84', '30 of 66'], 'takedowns': ['1 of 4', '0 of 2'], 'takedown percentage': ['25%', '0%'], 'submission attempts': ['1', '0'], 'reversals': ['0', '0'], 'control': ['3:16', '0:01']}


In [277]:
test_fight_stats['fight_totals']

{'fighters': ['Renato Moicano ', 'Chris Duncan '],
 'knockdowns': ['0', '0'],
 'significant strikes': ['21 of 36', '27 of 62'],
 'significant strike percentage': ['58%', '43%'],
 'total strike': ['62 of 84', '30 of 66'],
 'takedowns': ['1 of 4', '0 of 2'],
 'takedown percentage': ['25%', '0%'],
 'submission attempts': ['1', '0'],
 'reversals': ['0', '0'],
 'control': ['3:16', '0:01']}

In [278]:
split_combined_data_rows(test_fight_stats['fight_totals'])

{'fighters_a': 'Renato Moicano ',
 'fighters_b': 'Chris Duncan ',
 'knockdowns_a': '0',
 'knockdowns_b': '0',
 'significant strikes_a': '21 of 36',
 'significant strikes_b': '27 of 62',
 'significant strike percentage_a': '58%',
 'significant strike percentage_b': '43%',
 'total strike_a': '62 of 84',
 'total strike_b': '30 of 66',
 'takedowns_a': '1 of 4',
 'takedowns_b': '0 of 2',
 'takedown percentage_a': '25%',
 'takedown percentage_b': '0%',
 'submission attempts_a': '1',
 'submission attempts_b': '0',
 'reversals_a': '0',
 'reversals_b': '0',
 'control_a': '3:16',
 'control_b': '0:01'}

In [279]:
# function -- when given an event, get all fights for the given event, parse each fight dict, and put them into an overarching event dict



In [280]:
all_events_url="http://ufcstats.com/statistics/events/completed?page=all"

test_soup = make_soup(all_events_url)

all_event_links = get_all_event_details(test_soup)

test_event = all_event_links[1]

test_event_soup = make_soup(test_event)

fight_links_of_single_event = get_all_fight_details(test_event_soup)
# fight_links[:5]
# len(fight_links)
fight_links_of_single_event


['http://ufcstats.com/fight-details/d69bd8e53ec858dc',
 'http://ufcstats.com/fight-details/721642e7de112e9e',
 'http://ufcstats.com/fight-details/ad487d1c6141d90a',
 'http://ufcstats.com/fight-details/fef80c98b2fba70c',
 'http://ufcstats.com/fight-details/c27bec45d5a29a2c',
 'http://ufcstats.com/fight-details/7924f10aac3543de',
 'http://ufcstats.com/fight-details/0760bc02280574dd',
 'http://ufcstats.com/fight-details/af2a9d47ca8115e4',
 'http://ufcstats.com/fight-details/9c9c26679701b84d',
 'http://ufcstats.com/fight-details/74bb036ef0493bf7',
 'http://ufcstats.com/fight-details/e284a194fb4648a9',
 'http://ufcstats.com/fight-details/caf13346cbcfdb03',
 'http://ufcstats.com/fight-details/a5a749aa0061fdb4']

In [281]:
def get_fighters_names(fight_soup_obj):
    fighter_a = fight_soup_obj.find_all('h3',{"class":"b-fight-details__person-name"})[0].text.replace(" \n","").replace("\n","")
    fighter_b = fight_soup_obj.find_all('h3',{"class":"b-fight-details__person-name"})[1].text.replace(" \n","").replace("\n","")

    fighters_names = fighter_a + ' vs ' + fighter_b

    return fighters_names

def get_event_specific_data(event_soup_obj):

    ## add functionality to save event name

    event_details = [item.text for item in event_soup_obj.find_all('li',{"class":"b-list__box-list-item"})]

    ## extract the event date
    date_r = re.compile("Date")
    event_date = list(filter(date_r.search, event_details)) 
    ## raise error if list contains more than 1 element
    event_date_clean = event_date[0].replace("\n\n        ","").replace("\n    ","").replace("    "," ")


    ## extract the event location
    location_r = re.compile("Location")

    event_location = list(filter(location_r.search, event_details))
    ## raise error if list contains more than 1 element
    event_location_clean = event_location[0].replace("\n\n        ","").replace("\n    ","").replace("  \n  "," ")

    return event_date_clean, event_location_clean


get_event_specific_data(test_event_soup)

('Date: April 04, 2026', 'Location: Las Vegas, Nevada, USA')

In [285]:
## function that takes an event,
def get_full_event_data(event_soup_object):

    full_event_data_dict = {}

    ## gets the date and location of event
    event_date, event_location = get_event_specific_data(event_soup_object)
    full_event_data_dict['event_date'] = event_date
    full_event_data_dict['event_location'] = event_location

    ## uses the fight dictionaries for each fight in the event
    fight_links_of_single_event = get_all_fight_details(event_soup_object)

    for fight_link in fight_links_of_single_event:
        fight_soup_obj = make_soup(fight_link)

        fight_name = get_fighters_names(fight_soup_obj)
        fight_id = fight_link.split('/fight-details/')[-1]

        fight_stats = extract_fight_data(fight_soup_obj, totals_table_structure, sig_str_table_structure)
        fight_stats['fight_name'] = fight_name

        full_event_data_dict[fight_id] = fight_stats

    ## saves everything as a nested dictionary object
    return full_event_data_dict




In [286]:
def is_future_fight(event_soup):
    event_date_string = get_event_specific_data(event_soup)[0][6:]
    event_dt_object = datetime.strptime(event_date_string, "%B %d, %Y")
    event_date = event_dt_object.date()

    return date.today() < event_date


def get_event_data_from_links(event_link_set):

    event_history_data = {}

    for event_link in event_link_set:

        event_id = event_link.split('/event-details/')[-1]
        event_soup_obj = make_soup(event_link)

        ## if it is a future event, do not save any data
        if is_future_fight(event_soup_obj):
            event_history_data[event_id] = 'FUTURE FIGHT'

        ## otherwise, save the event data
        else:
            event_data = get_full_event_data(event_soup_obj)
            event_history_data[event_id] = event_data

    return event_history_data

test_event_set = all_event_links[0:11]
test_full_dat = get_event_data_from_links(test_event_set)

In [288]:
test_full_dat

{'f3eb664db7fb1df3': 'FUTURE FIGHT',
 '9a70f67ad2187fa3': {'event_date': 'Date: April 04, 2026',
  'event_location': 'Location: Las Vegas, Nevada, USA',
  'd69bd8e53ec858dc': {'fight_totals': {'fighters': ['Renato Moicano ',
     'Chris Duncan '],
    'knockdowns': ['0', '0'],
    'significant strikes': ['21 of 36', '27 of 62'],
    'significant strike percentage': ['58%', '43%'],
    'total strike': ['62 of 84', '30 of 66'],
    'takedowns': ['1 of 4', '0 of 2'],
    'takedown percentage': ['25%', '0%'],
    'submission attempts': ['1', '0'],
    'reversals': ['0', '0'],
    'control': ['3:16', '0:01']},
   'totals_per_round': {'Round 1': {'fighters': ['Renato Moicano ',
      'Chris Duncan '],
     'knockdowns': ['0', '0'],
     'significant strikes': ['14 of 20', '17 of 38'],
     'significant strike percentage': ['70%', '44%'],
     'total strike': ['16 of 22', '19 of 41'],
     'takedowns': ['0 of 2', '0 of 1'],
     'takedown percentage': ['0%', '0%'],
     'submission attempts':